# Regression — EV Charging Demand Prediction

## Step 1 — Load Dataset

In [1]:
import pandas as pd

df = pd.read_csv(
    "Data_row/cleaned data/ChargingRecords_Cleaned.csv"
)

df["StartDatetime"] = pd.to_datetime(df["StartDatetime"])

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (72843, 19)


,UserID,ChargerID,ChargerCompany,Location,ChargerType,StartDay,StartTime,EndDay,EndTime,StartDatetime,EndDatetime,Duration,Demand,CalculateDuration,PreciseDuration,InvalidNegativeDuration,ZeroDurationRecoverable,ZeroDurationUnresolved,CleanedDuration
0,0,1,1,hotel,0,2022-09-15,20:54:02,2022-09-15,23:59:13,2022-09-15 20:54:00,2022-09-15 23:59:00,185,20.36,185.0,185.183333,False,False,False,185.0
1,0,1,1,hotel,0,2022-09-14,20:01:05,2022-09-14,21:31:04,2022-09-14 20:01:00,2022-09-14 21:31:00,90,10.19,90.0,89.983333,False,False,False,90.0
2,0,1,1,hotel,0,2022-09-14,18:54:30,2022-09-14,19:54:29,2022-09-14 18:54:00,2022-09-14 19:54:00,60,6.78,60.0,59.983333,False,False,False,60.0
3,0,1,1,hotel,0,2022-09-29,18:32:51,2022-09-30,0:16:42,2022-09-29 18:32:00,2022-09-30 00:16:00,344,37.65,344.0,343.850000,False,False,False,344.0
4,0,1,1,hotel,0,2022-09-25,19:30:15,2022-09-26,0:30:14,2022-09-25 19:30:00,2022-09-26 00:30:00,300,33.81,300.0,299.983333,False,False,False,300.0


## Step 2 — Chronological Train-Test Split

In [2]:
# Sort charging sessions by date and time
df = df.sort_values("StartDatetime").reset_index(drop=True)

# Calculate 80% split position
split_index = int(len(df) * 0.8)

# Divide dataset
train_df = df.iloc[:split_index].copy()
test_df = df.iloc[split_index:].copy()

# Display results
print("Training rows:", len(train_df))
print("Testing rows:", len(test_df))

print("Training period:",
      train_df["StartDatetime"].min(),
      "to",
      train_df["StartDatetime"].max())

print("Testing period:",
      test_df["StartDatetime"].min(),
      "to",
      test_df["StartDatetime"].max())

Training rows: 58274
Testing rows: 14569
Training period: 2021-09-30 17:14:00 to 2022-07-30 18:55:00
Testing period: 2022-07-30 18:58:00 to 2022-09-30 23:11:00


## Step 3 — Prepare Features and Target

In [3]:
# Features available when charging starts
feature_columns = [
    "ChargerType",
    "Location",
    "ChargerCompany",
    "StartHour",
    "DayOfWeek",
    "Month",
    "IsWeekend"
]

# Create time features for both datasets
for data in [train_df, test_df]:
    data["StartHour"] = data["StartDatetime"].dt.hour
    data["DayOfWeek"] = data["StartDatetime"].dt.dayofweek
    data["Month"] = data["StartDatetime"].dt.month
    data["IsWeekend"] = (data["DayOfWeek"] >= 5).astype(int)

# Separate inputs and target
X_train = train_df[feature_columns]
X_test = test_df[feature_columns]

y_train = train_df["Demand"]
y_test = test_df["Demand"]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (58274, 7)
X_test: (14569, 7)
y_train: (58274,)
y_test: (14569,)


## Step 4 — Convert Categorical Features into Numbers

In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

# Select categorical columns
categorical_columns = [
    "ChargerType",
    "Location",
    "ChargerCompany"
]

# Create preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_columns
        )
    ],
    remainder="passthrough"
)

print("Preprocessor created successfully!")

Preprocessor created successfully!


## Step 5 — Train Linear Regression Model

In [5]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

# Create machine learning pipeline
linear_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Train the model
linear_model.fit(X_train, y_train)

# Predict charging demand
y_pred = linear_model.predict(X_test)

print("Linear Regression trained successfully!")
print("Total predictions:", len(y_pred))

Linear Regression trained successfully!
Total predictions: 14569


## Step 6 — Evaluate Linear Regression

In [6]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
import numpy as np

# Calculate evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

# Display results
print("Linear Regression Performance")
print("-----------------------------")
print("MAE :", round(mae, 4))
print("MSE :", round(mse, 4))
print("RMSE:", round(rmse, 4))
print("R²  :", round(r2, 4))

Linear Regression Performance
-----------------------------
MAE : 10.3957
MSE : 182.6671
RMSE: 13.5154
R²  : 0.0566


## Step 7 — Train Decision Tree Regression

In [7]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.pipeline import Pipeline

# Create Decision Tree pipeline
tree_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(
        max_depth=10,
        min_samples_leaf=10,
        random_state=42
    ))
])

# Train model
tree_model.fit(X_train, y_train)

# Predict demand
tree_pred = tree_model.predict(X_test)

# Evaluate model
tree_mae = mean_absolute_error(y_test, tree_pred)
tree_mse = mean_squared_error(y_test, tree_pred)
tree_rmse = np.sqrt(tree_mse)
tree_r2 = r2_score(y_test, tree_pred)

print("Decision Tree Regression Performance")
print("------------------------------------")
print("MAE :", round(tree_mae, 4))
print("MSE :", round(tree_mse, 4))
print("RMSE:", round(tree_rmse, 4))
print("R²  :", round(tree_r2, 4))

Decision Tree Regression Performance
------------------------------------
MAE : 10.156
MSE : 176.7558
RMSE: 13.295
R²  : 0.0871


## Step 8 — Train Random Forest Regression
Random Forest combines multiple decision trees to make predictions. It can often perform better than a single decision tree.

In [8]:
from sklearn.ensemble import RandomForestRegressor

# Create Random Forest pipeline
rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        max_depth=15,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Train model
rf_model.fit(X_train, y_train)

# Predict demand
rf_pred = rf_model.predict(X_test)

# Evaluate model
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_mse = mean_squared_error(y_test, rf_pred)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Regression Performance")
print("------------------------------------")
print("MAE :", round(rf_mae, 4))
print("MSE :", round(rf_mse, 4))
print("RMSE:", round(rf_rmse, 4))
print("R²  :", round(rf_r2, 4))

Random Forest Regression Performance
------------------------------------
MAE : 10.1109
MSE : 177.6567
RMSE: 13.3288
R²  : 0.0824


## Step 9 — Train Gradient Boosting Regression
Gradient Boosting builds decision trees one after another. Each new tree tries to correct errors made by the previous trees.

In [9]:
from sklearn.ensemble import GradientBoostingRegressor

# Create Gradient Boosting pipeline
gb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=42
    ))
])

# Train model
gb_model.fit(X_train, y_train)

# Predict demand
gb_pred = gb_model.predict(X_test)

# Evaluate model
gb_mae = mean_absolute_error(y_test, gb_pred)
gb_mse = mean_squared_error(y_test, gb_pred)
gb_rmse = np.sqrt(gb_mse)
gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting Regression Performance")
print("----------------------------------------")
print("MAE :", round(gb_mae, 4))
print("MSE :", round(gb_mse, 4))
print("RMSE:", round(gb_rmse, 4))
print("R²  :", round(gb_r2, 4))

Gradient Boosting Regression Performance
----------------------------------------
MAE : 10.0443
MSE : 173.0961
RMSE: 13.1566
R²  : 0.106


## Step 10 — Train XGBoost Regression

In [10]:
import xgboost

print("XGBoost version:", xgboost.__version__)

XGBoost version: 3.4.1


In [11]:
%pip install xgboost


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [12]:
from xgboost import XGBRegressor

# Create XGBoost pipeline
xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        n_jobs=-1
    ))
])

# Train model
xgb_model.fit(X_train, y_train)

# Predict demand
xgb_pred = xgb_model.predict(X_test)

# Evaluate model
xgb_mae = mean_absolute_error(y_test, xgb_pred)
xgb_mse = mean_squared_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(xgb_mse)
xgb_r2 = r2_score(y_test, xgb_pred)

print("XGBoost Regression Performance")
print("--------------------------------")
print("MAE :", round(xgb_mae, 4))
print("MSE :", round(xgb_mse, 4))
print("RMSE:", round(xgb_rmse, 4))
print("R²  :", round(xgb_r2, 4))

XGBoost Regression Performance
--------------------------------
MAE : 10.1632
MSE : 172.6865
RMSE: 13.141
R²  : 0.1081


## Step 11 — Train LightGBM Regression

In [13]:
from lightgbm import LGBMRegressor

# Create LightGBM pipeline
lgbm_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LGBMRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        num_leaves=20,
        random_state=42,
        verbosity=-1
    ))
])

# Train model
lgbm_model.fit(X_train, y_train)

# Predict demand
lgbm_pred = lgbm_model.predict(X_test)

# Evaluate model
lgbm_mae = mean_absolute_error(y_test, lgbm_pred)
lgbm_mse = mean_squared_error(y_test, lgbm_pred)
lgbm_rmse = np.sqrt(lgbm_mse)
lgbm_r2 = r2_score(y_test, lgbm_pred)

print("LightGBM Regression Performance")
print("-------------------------------")
print("MAE :", round(lgbm_mae, 4))
print("MSE :", round(lgbm_mse, 4))
print("RMSE:", round(lgbm_rmse, 4))
print("R²  :", round(lgbm_r2, 4))

LightGBM Regression Performance
-------------------------------
MAE : 9.9771
MSE : 170.937
RMSE: 13.0743
R²  : 0.1171


## Step 12 — Compare Regression Models

In [14]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting",
        "XGBoost",
        "LightGBM"
    ],
    "MAE": [mae, tree_mae, rf_mae, gb_mae, xgb_mae, lgbm_mae],
    "MSE": [mse, tree_mse, rf_mse, gb_mse, xgb_mse, lgbm_mse],
    "RMSE": [rmse, tree_rmse, rf_rmse, gb_rmse, xgb_rmse, lgbm_rmse],
    "R2": [r2, tree_r2, rf_r2, gb_r2, xgb_r2, lgbm_r2]
})

results = results.sort_values("RMSE").reset_index(drop=True)

results.round(4)

,Model,MAE,MSE,RMSE,R2
0,LightGBM,9.9771,170.9370,13.0743,0.1171
1,XGBoost,10.1632,172.6865,13.1410,0.1081
2,Gradient Boosting,10.0443,173.0961,13.1566,0.1060
3,Decision Tree,10.1560,176.7558,13.2950,0.0871
4,Random Forest,10.1109,177.6567,13.3288,0.0824
5,Linear Regression,10.3957,182.6671,13.5154,0.0566


## Step 13 — LightGBM Feature Importance

In [15]:
import pandas as pd

# Get feature names after encoding
feature_names = lgbm_model.named_steps[
    "preprocessor"
].get_feature_names_out()

# Get feature importance values
importance_values = lgbm_model.named_steps[
    "model"
].feature_importances_

# Create feature importance table
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": importance_values
})

# Display top 15 features
importance_df = importance_df.sort_values(
    "Importance", ascending=False
)

importance_df.head(15)

,Feature,Importance
18,remainder__StartHour,530
20,remainder__Month,314
19,remainder__DayOfWeek,174
16,categorical__ChargerCompany_0,148
0,categorical__ChargerType_0,109
9,categorical__Location_market,77
10,categorical__Location_public area,59
6,categorical__Location_company,54
13,categorical__Location_resort,53
3,categorical__Location_apartment,52


## Step 14 — Compare Actual vs Predicted Demand

In [16]:
# Compare actual and predicted demand
comparison_df = pd.DataFrame({
    "Actual Demand": y_test.to_numpy(),
    "Predicted Demand": lgbm_pred
})

# Calculate prediction errors
comparison_df["Error"] = (
    comparison_df["Actual Demand"]
    - comparison_df["Predicted Demand"]
)

comparison_df["Absolute Error"] = comparison_df["Error"].abs()

# Display first 10 predictions
comparison_df.head(10).round(2)

,Actual Demand,Predicted Demand,Error,Absolute Error
0,13.02,15.65,-2.63,2.63
1,3.53,18.67,-15.14,15.14
2,11.15,20.43,-9.28,9.28
3,49.02,24.91,24.11,24.11
4,2.16,8.36,-6.20,6.20
5,28.61,16.46,12.15,12.15
6,0.01,16.46,-16.45,16.45
7,29.34,16.46,12.88,12.88
8,12.13,21.80,-9.67,9.67
9,17.32,18.98,-1.66,1.66


## Step 15 — Check Prediction Bias

In [17]:
# Calculate average demand
actual_mean = comparison_df["Actual Demand"].mean()
predicted_mean = comparison_df["Predicted Demand"].mean()

# Calculate average prediction error
mean_error = comparison_df["Error"].mean()

print("Average Actual Demand:", round(actual_mean, 2))
print("Average Predicted Demand:", round(predicted_mean, 2))
print("Mean Error:", round(mean_error, 2))

Average Actual Demand: 17.74
Average Predicted Demand: 17.55
Mean Error: 0.19


## Step 16 — Check Model Overfitting

In [18]:
# Predict training demand
train_pred = lgbm_model.predict(X_train)

# Compare training and testing performance
print("LightGBM Training vs Testing")
print("----------------------------")

print("Training R²:", round(r2_score(y_train, train_pred), 4))
print("Testing R² :", round(r2_score(y_test, lgbm_pred), 4))

print("Training MAE:", round(mean_absolute_error(y_train, train_pred), 4))
print("Testing MAE :", round(mean_absolute_error(y_test, lgbm_pred), 4))

LightGBM Training vs Testing
----------------------------
Training R²: 0.1679
Testing R² : 0.1171
Training MAE: 9.1989
Testing MAE : 9.9771


## Step 17 — Check Charger Usage History

In [19]:
# Count charging sessions for each charger
charger_counts = train_df["ChargerID"].value_counts()

print("Total unique chargers:", charger_counts.size)

print("Chargers with more than 1 session:",
      (charger_counts > 1).sum())

print("Maximum sessions for one charger:",
      charger_counts.max())

print("Average sessions per charger:",
      round(charger_counts.mean(), 2))

Total unique chargers: 1964
Chargers with more than 1 session: 1687
Maximum sessions for one charger: 586
Average sessions per charger: 29.67


## Step 18 — Create Previous Charger Demand Feature

In [20]:
# Prepare chronological session history
history_df = df.sort_values("StartDatetime").copy()

history_df["EndDatetime"] = pd.to_datetime(
    history_df["EndDatetime"]
)

# Find previous session on the same charger
history_df["PreviousChargerDemand"] = (
    history_df.groupby("ChargerID")["Demand"].shift(1)
)

history_df["PreviousChargerEnd"] = (
    history_df.groupby("ChargerID")["EndDatetime"].shift(1)
)

# Only keep demand if the previous session had finished
valid_history = (
    history_df["PreviousChargerEnd"]
    <= history_df["StartDatetime"]
)

history_df.loc[
    ~valid_history, "PreviousChargerDemand"
] = float("nan")

# Check results
print(
    "Sessions with available previous charger demand:",
    history_df["PreviousChargerDemand"].notna().sum()
)

history_df[
    ["ChargerID", "StartDatetime", "PreviousChargerDemand"]
].head(10)

Sessions with available previous charger demand: 70642


,ChargerID,StartDatetime,PreviousChargerDemand
0,291,2021-09-30 17:14:00,NaN
1,191,2021-09-30 18:22:00,NaN
2,320,2021-09-30 18:57:00,NaN
3,433,2021-09-30 19:52:00,NaN
4,250,2021-09-30 20:48:00,NaN
5,248,2021-09-30 21:05:00,NaN
6,312,2021-09-30 21:08:00,NaN
7,260,2021-09-30 21:16:00,NaN
8,211,2021-09-30 21:39:00,NaN
9,64,2021-09-30 22:13:00,NaN


## Step 18 — Compare LightGBM with a Baseline Model

In [21]:
from sklearn.dummy import DummyRegressor

# Create baseline model
baseline_model = DummyRegressor(strategy="mean")

# Train baseline
baseline_model.fit(X_train, y_train)

# Predict demand
baseline_pred = baseline_model.predict(X_test)

# Evaluate baseline
baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_mse = mean_squared_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(baseline_mse)
baseline_r2 = r2_score(y_test, baseline_pred)

# Compare results
baseline_results = pd.DataFrame({
    "Model": ["Baseline", "LightGBM"],
    "MAE": [baseline_mae, lgbm_mae],
    "MSE": [baseline_mse, lgbm_mse],
    "RMSE": [baseline_rmse, lgbm_rmse],
    "R2": [baseline_r2, lgbm_r2]
})

baseline_results.round(4)

,Model,MAE,MSE,RMSE,R2
0,Baseline,10.8669,193.7579,13.9197,-0.0007
1,LightGBM,9.9771,170.9370,13.0743,0.1171


## Step 19 — Check Overlapping Charger Sessions

In [22]:
# Convert end time into datetime
df["EndDatetime"] = pd.to_datetime(df["EndDatetime"])

# Sort sessions by charger and start time
history_check = df.sort_values(
    ["ChargerID", "StartDatetime"]
).copy()

# Get previous session's end time
history_check["PreviousEnd"] = (
    history_check.groupby("ChargerID")["EndDatetime"].shift(1)
)

# Check overlapping sessions
overlap = (
    history_check["PreviousEnd"]
    > history_check["StartDatetime"]
)

print("Overlapping sessions:", overlap.sum())
print("Total sessions:", len(history_check))
print("Overlap percentage:", round(overlap.mean() * 100, 2))

Overlapping sessions: 83
Total sessions: 72843
Overlap percentage: 0.11


## Step 20 — Create Historical Charger Demand

In [23]:
# Sort sessions by start time
history_df = df.sort_values("StartDatetime").copy()

# Create an empty historical feature
history_df["PreviousChargerDemand"] = float("nan")

# Store completed charging sessions
completed_sessions = history_df[
    ["ChargerID", "EndDatetime", "Demand"]
].dropna().sort_values("EndDatetime")

# Match each session with the latest completed session
history_df = pd.merge_asof(
    history_df,
    completed_sessions,
    left_on="StartDatetime",
    right_on="EndDatetime",
    by="ChargerID",
    direction="backward",
    allow_exact_matches=True,
    suffixes=("", "_previous")
)

# Store previous completed demand
history_df["PreviousChargerDemand"] = history_df["Demand_previous"]

# Remove temporary columns
history_df = history_df.drop(
    columns=["EndDatetime_previous", "Demand_previous"]
)

print(
    "Sessions with historical demand:",
    history_df["PreviousChargerDemand"].notna().sum()
)

history_df[
    ["ChargerID", "StartDatetime", "PreviousChargerDemand"]
].head(10)

Sessions with historical demand: 70759


,ChargerID,StartDatetime,PreviousChargerDemand
0,291,2021-09-30 17:14:00,NaN
1,191,2021-09-30 18:22:00,NaN
2,320,2021-09-30 18:57:00,NaN
3,433,2021-09-30 19:52:00,NaN
4,250,2021-09-30 20:48:00,NaN
5,248,2021-09-30 21:05:00,NaN
6,312,2021-09-30 21:08:00,NaN
7,260,2021-09-30 21:16:00,NaN
8,211,2021-09-30 21:39:00,NaN
9,64,2021-09-30 22:13:00,NaN



## Step 21 — Prepare Historical Features

In [24]:
# Restore chronological order
history_df = history_df.sort_values(
    "StartDatetime"
).reset_index(drop=True)

# Use the original 80/20 split
train_history = history_df.iloc[:split_index].copy()
test_history = history_df.iloc[split_index:].copy()

# Create time features
for data in [train_history, test_history]:
    data["StartHour"] = data["StartDatetime"].dt.hour
    data["DayOfWeek"] = data["StartDatetime"].dt.dayofweek
    data["Month"] = data["StartDatetime"].dt.month
    data["IsWeekend"] = (data["DayOfWeek"] >= 5).astype(int)

# Add historical demand to existing features
history_features = feature_columns + [
    "PreviousChargerDemand"
]

# Prepare inputs and target
X_train_history = train_history[history_features]
X_test_history = test_history[history_features]

y_train_history = train_history["Demand"]
y_test_history = test_history["Demand"]

print("Training shape:", X_train_history.shape)
print("Testing shape:", X_test_history.shape)

print(
    "Missing historical values in training:",
    X_train_history["PreviousChargerDemand"].isna().sum()
)

print(
    "Missing historical values in testing:",
    X_test_history["PreviousChargerDemand"].isna().sum()
)

Training shape: (58274, 8)
Testing shape: (14569, 8)
Missing historical values in training: 1929
Missing historical values in testing: 155


## Step 22 — LightGBM with Historical Demand

In [25]:
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from lightgbm import LGBMRegressor

# Categorical and numerical features
categorical_features = [
    "ChargerType",
    "Location",
    "ChargerCompany"
]

numerical_features = [
    "StartHour",
    "DayOfWeek",
    "Month",
    "IsWeekend",
    "PreviousChargerDemand"
]

# Preprocessing
history_preprocessor = ColumnTransformer([
    ("categorical",
     OneHotEncoder(handle_unknown="ignore"),
     categorical_features),

    ("numerical",
     SimpleImputer(strategy="median"),
     numerical_features)
])

# Create LightGBM pipeline
history_model = Pipeline([
    ("preprocessor", history_preprocessor),
    ("model", LGBMRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        num_leaves=20,
        random_state=42,
        verbosity=-1
    ))
])

# Train and predict
history_model.fit(X_train_history, y_train_history)

history_pred = history_model.predict(X_test_history)

# Evaluate
history_mae = mean_absolute_error(y_test_history, history_pred)
history_mse = mean_squared_error(y_test_history, history_pred)
history_rmse = np.sqrt(history_mse)
history_r2 = r2_score(y_test_history, history_pred)

print("LightGBM with Historical Demand")
print("-------------------------------")
print("MAE :", round(history_mae, 4))
print("MSE :", round(history_mse, 4))
print("RMSE:", round(history_rmse, 4))
print("R²  :", round(history_r2, 4))

LightGBM with Historical Demand
-------------------------------
MAE : 9.5341
MSE : 159.5381
RMSE: 12.6308
R²  : 0.176


## Step 23 — Validate Historical Demand

In [26]:
# Check for invalid charging timestamps
invalid_timestamps = (
    df["EndDatetime"] < df["StartDatetime"]
)

print("Sessions with invalid timestamps:",
      invalid_timestamps.sum())

# Check historical feature availability
print("Sessions with historical demand:",
      history_df["PreviousChargerDemand"].notna().sum())

print("Sessions without historical demand:",
      history_df["PreviousChargerDemand"].isna().sum())

Sessions with invalid timestamps: 17
Sessions with historical demand: 70759
Sessions without historical demand: 2084


## Step 24 — Correct Historical Demand

In [27]:
# Keep only valid completed sessions
valid_sessions = df[
    (df["EndDatetime"] >= df["StartDatetime"])
    & df["EndDatetime"].notna()
    & df["StartDatetime"].notna()
].copy()

# Prepare valid historical records
completed_sessions = valid_sessions[
    ["ChargerID", "EndDatetime", "Demand"]
].sort_values("EndDatetime")

# Prepare current sessions
history_df = df.sort_values("StartDatetime").copy()

# Match with the latest valid completed session
history_df = pd.merge_asof(
    history_df,
    completed_sessions,
    left_on="StartDatetime",
    right_on="EndDatetime",
    by="ChargerID",
    direction="backward",
    allow_exact_matches=False,
    suffixes=("", "_previous")
)

# Save corrected historical feature
history_df["PreviousChargerDemand"] = (
    history_df["Demand_previous"]
)

# Remove temporary columns
history_df = history_df.drop(
    columns=["EndDatetime_previous", "Demand_previous"]
)

print("Invalid sessions excluded from history:",
      len(df) - len(valid_sessions))

print("Sessions with valid historical demand:",
      history_df["PreviousChargerDemand"].notna().sum())

print("Sessions without historical demand:",
      history_df["PreviousChargerDemand"].isna().sum())

Invalid sessions excluded from history: 17
Sessions with valid historical demand: 70711
Sessions without historical demand: 2132


## Step 25 — Retrain LightGBM with Corrected History

In [28]:
# Sort corrected historical data
history_df = history_df.sort_values(
    "StartDatetime"
).reset_index(drop=True)

# Recreate training and testing datasets
train_history = history_df.iloc[:split_index].copy()
test_history = history_df.iloc[split_index:].copy()

# Create time features
for data in [train_history, test_history]:
    data["StartHour"] = data["StartDatetime"].dt.hour
    data["DayOfWeek"] = data["StartDatetime"].dt.dayofweek
    data["Month"] = data["StartDatetime"].dt.month
    data["IsWeekend"] = (data["DayOfWeek"] >= 5).astype(int)

# Prepare model inputs
X_train_history = train_history[history_features]
X_test_history = test_history[history_features]

y_train_history = train_history["Demand"]
y_test_history = test_history["Demand"]

# Retrain LightGBM
history_model.fit(X_train_history, y_train_history)

# Make new predictions
history_pred = history_model.predict(X_test_history)

# Calculate corrected results
history_mae = mean_absolute_error(y_test_history, history_pred)
history_mse = mean_squared_error(y_test_history, history_pred)
history_rmse = np.sqrt(history_mse)
history_r2 = r2_score(y_test_history, history_pred)

print("Corrected Historical LightGBM Results")
print("-------------------------------------")
print("MAE :", round(history_mae, 4))
print("MSE :", round(history_mse, 4))
print("RMSE:", round(history_rmse, 4))
print("R²  :", round(history_r2, 4))

Corrected Historical LightGBM Results
-------------------------------------
MAE : 9.5494
MSE : 159.9609
RMSE: 12.6476
R²  : 0.1738


## Step 26 — Check Historical Data Leakage

In [29]:
# Give each session a unique row ID
sessions = df.reset_index().rename(
    columns={"index": "SessionID"}
)

# Keep only sessions with valid timestamps
valid_sessions = sessions[
    sessions["EndDatetime"].notna()
    & sessions["StartDatetime"].notna()
    & (sessions["EndDatetime"] >= sessions["StartDatetime"])
].copy()

# Prepare completed sessions
completed = valid_sessions[
    ["SessionID", "ChargerID", "EndDatetime", "Demand"]
].sort_values("EndDatetime")

# Match previous completed sessions
check_df = pd.merge_asof(
    sessions.sort_values("StartDatetime"),
    completed,
    left_on="StartDatetime",
    right_on="EndDatetime",
    by="ChargerID",
    direction="backward",
    allow_exact_matches=False,
    suffixes=("", "_previous")
)

# Check whether any session matched itself
self_matches = (
    check_df["SessionID"]
    == check_df["SessionID_previous"]
)

print("Historical self-matches:", self_matches.sum())

Historical self-matches: 0


## Step 27 — Save Regression Model

In [30]:
import os
import joblib

# Create folder for trained models
os.makedirs("models", exist_ok=True)

# Save LightGBM pipeline
joblib.dump(
    history_model,
    "models/lightgbm_regression.pkl"
)

print("Regression model saved successfully!")

Regression model saved successfully!
